# 08 — Brain states × mapas de referência

Interpreta uma definição de cluster (um conjunto de brain states) em termos neuroanatômicos: a ativação média de cada estado no córtex e no subcórtex, e a correlação com mapas **moleculares** (PET, neuromaps), **comportamentais** (taxonomia BrainMap × Brainnetome) e **canônicos** (Yeo 7 e as nossas 14 redes).

Duas etapas, dois scripts nesta pasta (`notebooks/`):

1. `state_correlations.compute(...)` → **tabelas** (`state_means.csv`, `correlations.csv`, `provenance.json`) em `results/<estados>/<atlas>/`
2. `state_figures.draw(...)` → **figuras** a partir dessas tabelas (visão geral + um card por estado)

Mapas de referência: rode uma vez `notebooks/fetch_reference_maps.sh`. Detalhes e limitações: `notebooks/state_maps.md`.

In [ ]:
import sys
from pathlib import Path
from IPython.display import Image, display

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "notebooks"))   # refmaps, state_correlations, state_figures
import state_correlations as sc
import state_figures as sf

## Parâmetros

**Definição de cluster** — `STATES`:
- `"simulate"`: seis estados simulados com resposta conhecida (para testar);
- uma coluna de estados do stage 4b, ex. `"HMM2_pca14_10"` (precisa de `OUTPUT_ROOT` e `WINDOW_S`);
- ou o caminho de um CSV escrito por `fmri-decomp state-means --csv`.

**Atlas** — `ATLASES`: onde os estados foram ajustados (`harvardoxford`, `networks_nodes`, `networks`, `yeo7`). Uma coluna de estados existe num atlas só; liste os atlas onde ela existe.

**O que mostrar** — `FAMILIES` (painéis não comportamentais) e `BN_LEVELS`: profundidade da taxonomia do BrainMap — `"domain"` (Action, Cognition, Emotion, Interoception, Perception), `"subcategory"` (ex. Vision · Motion), ou os dois.

**Estatística** — `N_NULLS` substitutos por mapa (`NULL="moran"`, segundos; `"burt"`, brainsmash, horas). Atlas com < 30 parcelas (`networks`, `yeo7`) ficam sem p-valor.

In [ ]:
STATES      = "simulate"            # "simulate" | "HMM2_pca14_10" | "path/to/state_means.csv"
WINDOW_S    = None                  # obrigatório p/ coluna de estados: -1 = per-TR (activation)
COHORTS     = None                  # None = todas as coortes da célula, juntas
OUTPUT_ROOT = None                  # None = $FMRIDECOMP_OUTPUTS ou output_root de config/camcan_movie.yaml

ATLASES     = ["harvardoxford"]     # + "networks_nodes", "networks", "yeo7"
FAMILIES    = ["molecular", "yeo7", "networks14"]
BN_LEVELS   = ["domain", "subcategory"]     # profundidade da taxonomia comportamental
BEHAVIOURAL = "neurosynth"          # perfis calculados do Neurosynth | CSV BrainMap real | None = simulado

N_NULLS     = 10_000
NULL        = "moran"
RESULTS     = REPO / "notebooks" / "state_maps_results"

## 1. Tabelas de correlação

In [ ]:
dirs = sc.compute(
    STATES, ATLASES,
    families=FAMILIES + [sc.BN_LEVELS[b] for b in BN_LEVELS],
    out=RESULTS, output_root=OUTPUT_ROOT, window_s=WINDOW_S, cohorts=COHORTS,
    behavioural_csv=BEHAVIOURAL, n_nulls=N_NULLS, null=NULL,
)
run_dir = next(iter(dirs.values())).parent
run_dir

In [ ]:
import pandas as pd
cor = pd.concat({a: pd.read_csv(d / "correlations.csv") for a, d in dirs.items()}, names=["atlas"])
cor.sort_values("p_null").groupby(["atlas", "family"]).head(3)

## 2. Figuras

In [ ]:
paths = sf.draw(run_dir, atlases=ATLASES, families=FAMILIES, bn_levels=BN_LEVELS)
for p in paths:
    if p.name == "state_maps_overview.png":
        display(Image(filename=str(p)))

In [ ]:
# um card por estado, em tamanho de leitura (todos os atlas pedidos)
for atlas in ATLASES:
    for card in sorted(dirs[atlas].glob("state_card_S*.png"),
                       key=lambda p: int(p.stem.rsplit("S", 1)[1])):
        print(atlas, card.stem)
        display(Image(filename=str(card)))